# ML Notes
A collection of machine learning concepts and code examples using NumPy, SciPy, Matplotlib, and scikit-learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import sklearn
from sklearn.metrics import r2_score

## 1. Prerequisite — Random Normal Distribution & Scatter Plot
`np.random.normal(mean, std, size)` generates normally distributed data. `plt.scatter(x, y)` plots it.

In [ ]:
x = np.random.normal(5, 2, 1000)  # np.random.normal(mean, std, size)
y = np.random.normal(100, 10, 1000)

plt.scatter(x, y)
plt.xlabel('x axis (maybe age of car)')
plt.ylabel('y axis (maybe speed of car)')
plt.show()

## 2. Linear Regression
`stats.linregress` gives slope, intercept, and `r` (correlation coefficient). `r` ranges from -1 to 1; 0 = no relation, ±1 = perfect relation.

In [ ]:
x = [5,7,8,7,2,17,2,9,4,11,12,9,6]
y = [99,86,87,88,111,86,103,87,94,78,77,85,86]

slope, intercept, r, p, stdErr = stats.linregress(x, y)
# r is the coefficient of correlation. range: (-1,1)
# 0 means no relation, 1/-1 means 100% related
print('r =', r)  # -0.76: there is a relation but not strong

def regPoints(x):
    return slope * x + intercept

regressionLine = list(map(regPoints, x))

# Prediction test
print('Speed of a 7yo car (predicted):', regPoints(7))

plt.scatter(x, y)
plt.plot(x, regressionLine)
plt.xlabel('age')
plt.ylabel('speed')
plt.grid()
plt.show()

## 3. Polynomial Regression
`np.polyfit(x, y, degree)` fits a polynomial curve. `np.poly1d(coeffs)` wraps those coefficients into a callable function f(x). Use `r2_score` to evaluate fit quality (range: (-∞, 1], 1 = perfect).

In [ ]:
x = [1,2,3,5,6,7,8,9,10,12,13,14,15,16,18,19,21,22]
y = [100,90,80,60,60,55,60,65,70,70,75,76,78,79,90,99,99,100]

# Degree 3 as the data bends twice
mymodel = np.poly1d(np.polyfit(x, y, 3))
# np.polyfit returns [a,b,c,d] coefficients
# np.poly1d wraps them into f(x) = ax³ + bx² + cx + d

print('R² score:', r2_score(y, mymodel(x)))

myline = np.linspace(x[0], x[-1], 100)  # 100 smooth x-values for plotting

# Prediction test
print('Predicted value at x=25:', mymodel(25))

plt.scatter(x, y)
plt.plot(myline, mymodel(myline))
plt.grid()
plt.show()

## 4. Multiple Regression
Multiple independent variables predicting one dependent variable using `sklearn.linear_model.LinearRegression`.

In [ ]:
df = pd.read_csv('data.csv')

X = df[['Weight', 'Volume']]  # independent vars
y = df['CO2']                 # dependent var

regr = sklearn.linear_model.LinearRegression()
regr.fit(X, y)
print('Coefficients:', regr.coef_)  # y = a*x + b

# Prediction test
predictedCO2 = regr.predict([[3300, 1300]])
print('Predicted CO2:', predictedCO2)

## 5. Feature Scaling
`StandardScaler` standardizes features (zero mean, unit variance). The model must be trained AND predicted on scaled inputs — use `.fit_transform()` on training data and `.transform()` on new inputs.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

df = pd.read_csv('data.csv')
df['Volume'] = df['Volume'] / 1000  # convert to cm³

scale = StandardScaler()

X = df[['Weight', 'Volume']]
y = df['CO2']

# .fit_transform = .fit (learns mean & std) + .transform (standardizes)
scaledX = scale.fit_transform(X)

myModel = LinearRegression()
myModel.fit(scaledX, y)  # trained on scaled features

weight_kg = 2300
vol_cm3 = 1.3

# Must scale new input the same way — .transform only (don't relearn)
scaledinput = scale.transform([[weight_kg, vol_cm3]])
predictedCO2 = myModel.predict(scaledinput)
print('Predicted CO2:', predictedCO2)

## 6. Train / Test Split
80% train, 20% test. Compare `r2_score` on both sets — similar scores mean the model generalizes well. Big gap = overfitting.

In [ ]:
np.random.seed(2)  # reproducibility

x = np.random.normal(3, 1, 100)
y = np.random.normal(150, 40, 100) / x

trainx = x[:80]
trainy = y[:80]
testx  = x[80:]
testy  = y[80:]

myModel = np.poly1d(np.polyfit(trainx, trainy, 4))

print('Train R²:', r2_score(trainy, myModel(trainx)))
print('Test R²: ', r2_score(testy,  myModel(testx)))

print('Prediction at x=5:', myModel(5))  # predicted ~22.8, real ~24

myLine = np.linspace(0, 6, 100)
plt.scatter(trainx, trainy)
plt.plot(myLine, myModel(myLine))
plt.show()

## 7. Decision Tree (Classification)
All columns must be numeric. `DecisionTreeClassifier` is non-deterministic by default (different tree each run). It learns split rules automatically from the features.

In [ ]:
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv('data_classification.csv')

# Encode categorical columns to numeric
df['Nationality'] = df['Nationality'].map({'UK': 0, 'USA': 1, 'N': 2})
df['Go'] = df['Go'].map({'YES': 1, 'NO': 0})

features = ['Age', 'Experience', 'Rank', 'Nationality']
X = df[features]
y = df['Go']

dtree = DecisionTreeClassifier()
dtree.fit(X, y)

# Should I go see a 40yo American comedian, 10yr experience, rank 7?
print(dtree.predict([[40, 10, 7, 1]]))  # [1] = go, [0] = no go

tree.plot_tree(dtree, feature_names=features)
plt.show()

## 8. Confusion Matrix & Evaluation Metrics
| Quadrant | Meaning |
|---|---|
| Top-Left | True Negative |
| Top-Right | False Positive |
| Bottom-Left | False Negative |
| Bottom-Right | True Positive |

- **Accuracy** = (TP + TN) / Total
- **Precision** = TP / (TP + FP)
- **Recall/Sensitivity** = TP / (TP + FN)
- **Specificity** = TN / (TN + FP)
- **F1** = 2 × (Precision × Recall) / (Precision + Recall)

In [ ]:
from sklearn import metrics

actual    = np.random.binomial(1, .9, size=1000)
predicted = np.random.binomial(1, .9, size=1000)

confusion_matrix = metrics.confusion_matrix(actual, predicted)
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=confusion_matrix)
cm_display.plot()
plt.show()

print({
    'Accuracy':          metrics.accuracy_score(actual, predicted),
    'Precision':         metrics.precision_score(actual, predicted),
    'Sensitivity/Recall':metrics.recall_score(actual, predicted),
    'Specificity':       metrics.recall_score(actual, predicted, pos_label=0),
    'F1_score':          metrics.f1_score(actual, predicted)
})

## 9. Hierarchical Clustering
Bottom-up (agglomerative): starts with each point as its own cluster, repeatedly merges the two closest clusters. `ward` linkage minimizes within-cluster variance.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

x = [4, 5, 10, 4, 3, 11, 14, 6, 10, 12]
y = [21, 19, 24, 17, 16, 25, 24, 22, 21, 21]

data = list(zip(x, y))  # 2D array of (x, y) pairs

hclust = AgglomerativeClustering(n_clusters=5, linkage='ward')
labels = hclust.fit_predict(data)

plt.scatter(x, y, c=labels)
plt.show()

## 10. Logistic Regression
Used for classification (not regression despite the name). Uses the sigmoid function to output probabilities. `coef_` is the weight `w` in `log_odds = wX + b`. If odds > 0.5, predict class 1.

In [ ]:
from sklearn import linear_model

X_raw = np.array([3.78, 2.44, 2.09, 0.14, 1.72, 1.65, 4.92, 4.37, 4.96, 4.52, 3.69, 5.88])
y     = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1])

df = pd.DataFrame({'TumorSize(cm)': X_raw, 'Is_Cancerous': y})

X = df[['TumorSize(cm)']]
y = df['Is_Cancerous']

logistic_reg = linear_model.LogisticRegression()
logistic_reg.fit(X, y)

print('Prediction for 3.46cm tumor:', logistic_reg.predict([[3.46]]))

log_odds = logistic_reg.coef_
odds = np.exp(log_odds)
print('Odds multiplier per 1cm increase:', odds)
# If odds = ~4, tumor size increasing by 1cm makes it 4x more likely to be cancerous

def logit2prob(model, X):
    log_odds = model.coef_ * X + model.intercept_  # log(p / (1-p))
    odds = np.exp(log_odds)                         # p / (1-p)
    return odds / (1 + odds)                        # p

df['Probability'] = logit2prob(logistic_reg, df[['TumorSize(cm)']])
print(df)
# .predict returns 0 or 1 based on probability threshold of 0.5

## 11. Grid Search (Hyperparameter Tuning)
Manually searching over hyperparameter values to maximize accuracy. `C` = inverse regularization strength (C = 1/λ). Higher C = less regularization. Combine with cross-validation to avoid overfitting.

In [ ]:
from sklearn import datasets
from sklearn.linear_model import LogisticRegression

iris = datasets.load_iris()
X = iris['data']
y = iris['target']

logit = LogisticRegression(max_iter=1000)  # default C=1
logit.fit(X, y)
print('Default C=1 accuracy:', logit.score(X, y))

C_values = [0.25, 0.5, 0.75, 1, 1.25, 1.5, 1.75, 2]
scores = []

for c in C_values:
    logit.set_params(C=c)
    logit.fit(X, y)
    scores.append(logit.score(X, y))

print(list(zip(C_values, scores)))
# Accuracy peaks around C=1.75 for this dataset

## 12. K-Means Clustering
**How it works:** Randomly assign each point to one of k clusters → calculate centroids → reassign points to nearest centroid → repeat until stable.

**Elbow Method:** Plot inertia (sum of squared distances to centroid) vs k. The 'elbow' point is the optimal k. Beyond that, adding clusters gives diminishing returns on inertia.

In [ ]:
from sklearn.cluster import KMeans

x = [4, 5, 10, 4, 3, 11, 14, 6, 10, 12]
y = [21, 19, 24, 17, 16, 25, 24, 22, 21, 21]

data = list(zip(x, y))
inertias = []

for i in range(1, len(y) + 1):
    kmeans = KMeans(n_clusters=i, n_init='auto')
    kmeans.fit(data)
    inertias.append(kmeans.inertia_)

# Elbow plot
plt.subplot(1, 2, 1)
plt.plot(range(1, len(x) + 1), inertias, '-o')
plt.title('Elbow Method')
plt.xlabel('No. of clusters')
plt.ylabel('Inertia')
# After k=2, the inertia drop slows — 2 is our elbow

# Final clustering with optimal k
k = 2
kmeans = KMeans(n_clusters=k, n_init='auto')
kmeans.fit(data)

plt.subplot(1, 2, 2)
plt.scatter(x, y, c=kmeans.labels_)
plt.title('Final clustering (k=2)')
plt.tight_layout()
plt.show()

## 13. KNN Classifier
`KNeighborsClassifier` classifies a point based on the majority class of its k nearest neighbours. Higher k = more robust to outliers but can overfit. Works on the principle that nearby observations are similar.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

x1      = [4, 5, 10, 4, 3, 11, 14, 8, 10, 12]
x2      = [21, 19, 24, 17, 16, 25, 24, 22, 21, 21]
classes = [0, 0, 1, 0, 0, 1, 1, 0, 1, 1]

df = pd.DataFrame({'in1': x1, 'in2': x2, 'class': classes})

plt.subplot(1, 2, 1)
plt.scatter(df['in1'], df['in2'], c=df['class'])
plt.title('Training data')

X = df[['in1', 'in2']]
y = df[['class']]

knn = KNeighborsClassifier(n_neighbors=5)  # change to 1 for stricter boundaries
knn.fit(X, y)

new_x1, new_x2 = 8, 21
prediction = knn.predict([[new_x1, new_x2]])
print('Predicted class:', prediction)

plt.subplot(1, 2, 2)
plt.scatter(x1 + [new_x1], x2 + [new_x2], c=classes + [prediction[0]])
plt.text(x=new_x1 - 1.7, y=new_x2 - 0.7, s=f'new point, class: {prediction[0]}')
plt.title('With new point')
plt.show()

## 14. KNN Regressor
`KNeighborsRegressor` predicts a continuous value by averaging the targets of the k nearest neighbours. Use odd k to avoid ties (more critical in classification). Pass feature names consistently to avoid sklearn warnings.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

x1      = [4, 5, 10, 4, 3, 11, 14, 8, 10, 12]
x2      = [21, 19, 24, 17, 16, 25, 24, 22, 21, 21]
targets = [10.5, 12.0, 25.5, 9.8, 8.5, 28.0, 30.2, 15.0, 22.1, 24.5]

df = pd.DataFrame({'in1': x1, 'in2': x2, 'target': targets})

plt.subplot(1, 2, 1)
plt.scatter(df['in1'], df['in2'], c=df['target'], cmap='viridis')
plt.title('Training Data Distribution')

X = df[['in1', 'in2']]  # pass as list of strings (best practice)
y = df['target']         # Series, not DataFrame, for regression

knn = KNeighborsRegressor(n_neighbors=3)  # odd k avoids ties
knn.fit(X, y)

# Match feature names during prediction
new_point = pd.DataFrame([[8, 21]], columns=['in1', 'in2'])
prediction = knn.predict(new_point)
print(f'Predicted Value: {prediction[0]:.2f}')

plt.subplot(1, 2, 2)
plt.scatter(df['in1'], df['in2'], c=df['target'], cmap='viridis', alpha=0.5)
plt.scatter(8, 21, c='red', marker='x', s=100)
plt.text(6.3, 20, f'Pred: {prediction[0]:.2f}', color='red', fontweight='bold')
plt.title('Prediction (k=3)')
plt.tight_layout()
plt.show()

## 15. Categorical Data — One-Hot Encoding
Models can't process strings. `OneHotEncoder` inside a `ColumnTransformer` converts categorical columns to binary columns while leaving numeric columns alone (`remainder='passthrough'`). The fitted `ct` remembers the encoding so `.transform()` works consistently on new inputs.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression

cars = pd.read_csv('data.csv')  # has 'Weight', 'Volume', 'Car', 'CO2' columns
X = cars[['Weight', 'Volume', 'Car']]  # 'Car' is still a string here
y = cars['CO2']

# Apply OneHotEncoder to 'Car', leave other columns as-is
ct = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(), ['Car'])],
    remainder='passthrough'
)

X_encoded = ct.fit_transform(X)
regr = LinearRegression()
regr.fit(X_encoded, y)

# Predicting with a string value — ct handles the encoding
sample = pd.DataFrame({'Weight': [2300], 'Volume': [1300], 'Car': ['VW']})
sample_encoded = ct.transform(sample)  # .transform only, don't refit

print('Predicted CO2:', regr.predict(sample_encoded))

## 16. Cross Validation
**K-Fold:** Splits data into k folds; trains on k-1, tests on the remaining 1, rotating through all folds. Gives k accuracy scores — average them for a reliable estimate.

**Stratified K-Fold:** Like K-Fold but ensures each fold has the same class proportion. Better for imbalanced classification datasets.

In [ ]:
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

X, y = datasets.load_iris(return_X_y=True)  # return_X_y returns X and y directly

clf = DecisionTreeClassifier(random_state=42)  # random_state removes dtree randomness

# --- K-Fold ---
k_folds = KFold(n_splits=5)
scores = cross_val_score(clf, X, y, cv=k_folds)
print('K-Fold CV Scores:', scores)
print('Average CV Score:', scores.mean())
print('Number of folds:', len(scores))

print()

# --- Stratified K-Fold (for imbalanced datasets) ---
# Ensures each fold has equal class proportion — better for classification
sk_folds = StratifiedKFold(n_splits=5)
scores = cross_val_score(clf, X, y, cv=sk_folds)
print('Stratified K-Fold CV Scores:', scores)
print('Average CV Score:', scores.mean())
print('Number of folds:', len(scores))